# 05 — Interview Questions

Purpose:
Practice realistic SQL interview questions using PostgreSQL telemetry and capacity data.

This notebook covers:
- PostgreSQL connection
- smoke test
- helper function to run SQL
- table inspection helper
- service capacity summaries
- threshold breaches
- noisy service detection
- daily and hourly rollups
- CTEs
- window functions
- incident-style troubleshooting
- interview explanations


## Cell 2 — Install/import dependencies


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Imports loaded.")


## Cell 3 — Connection settings


In [ ]:
DB_HOST = "host.docker.internal"
DB_PORT = 5432
DB_NAME = "studybook"
DB_USER = "sb_user"
DB_PASSWORD = "sb_pass_123"

password_encoded = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{password_encoded}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database URL created.")

# If this notebook runs directly on Windows instead of inside a container,
# change DB_HOST to "localhost".


## Cell 4 — Smoke test connection


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, now();"))
    row = result.fetchone()

row


## Cell 5 — Helper function to run SQL


In [ ]:
def run_sql(sql: str) -> pd.DataFrame:
    """
    Run SQL against the local PostgreSQL telemetry lab
    and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)


## Cell 6 — Helper function to inspect one table safely


In [ ]:
def inspect_table_safe(table_name: str) -> None:
    """
    Safely inspect a table without changing data.
    Shows column metadata, row count, and a small preview.
    """
    metadata_sql = f"""
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = '{table_name}'
    ORDER BY ordinal_position;
    """

    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM {table_name};
    """

    preview_sql = f"""
    SELECT *
    FROM {table_name}
    LIMIT 10;
    """

    print(f"Column metadata for public.{table_name}")
    display(run_sql(metadata_sql))

    print(f"Row count for public.{table_name}")
    display(run_sql(count_sql))

    print(f"Preview rows from public.{table_name}")
    display(run_sql(preview_sql))


# 05 — Interview Question Practice


## 05.1 Verify tables used in this notebook

This notebook uses:
- `telemetry_samples` as the metric fact table
- `services` as the service lookup table
- `hosts` as the host lookup table when needed
- `incidents` if it exists
- `deployments` if it exists
- `capacity_thresholds` if it exists


In [ ]:
sql = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name IN (
      'telemetry_samples',
      'services',
      'hosts',
      'incidents',
      'deployments',
      'capacity_thresholds'
  )
ORDER BY table_name;
"""

run_sql(sql)


## 05.2 Inspect telemetry_samples

`telemetry_samples` contains sampled telemetry for CPU, memory, latency,
request rate, error rate, resource allocation, forecast values, cost,
and JSONB tags.


In [ ]:
inspect_table_safe("telemetry_samples")


## 05.3 Question 1 — Which services have the highest average CPU?

This is a basic JOIN + GROUP BY interview question.
It summarizes raw telemetry into service-level capacity findings.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(t.cpu_utilization_pct), 2) AS max_cpu_pct,
    COUNT(*) AS sample_count
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY avg_cpu_pct DESC;
"""

run_sql(sql)


## 05.4 Question 2 — Which services have the highest memory pressure?

This is similar to CPU analysis, but focused on memory.
Memory pressure can be more dangerous because memory exhaustion may cause crashes or restarts.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(MAX(t.memory_utilization_pct), 2) AS max_memory_pct,
    COUNT(*) AS sample_count
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY avg_memory_pct DESC;
"""

run_sql(sql)


## 05.5 Question 3 — Which services have risky P95 latency?

`p95_latency_ms` is already a sampled P95 metric.
This query calculates P95 across sampled P95 values per service.
Call it P95 of sampled P95 latency, not true raw request-level P95.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(AVG(t.p95_latency_ms), 2) AS avg_sampled_p95_latency_ms,
    MAX(t.p95_latency_ms) AS max_sampled_p95_latency_ms,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.p95_latency_ms)::NUMERIC,
        2
    ) AS p95_of_sampled_p95_latency_ms
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY p95_of_sampled_p95_latency_ms DESC;
"""

run_sql(sql)


## 05.6 Question 4 — Which samples crossed risk thresholds?

This looks for individual telemetry samples that crossed CPU, memory, latency,
error, or forecast thresholds.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    s.service_name,
    t.host_id,
    t.cpu_utilization_pct,
    t.memory_utilization_pct,
    t.p95_latency_ms,
    t.error_rate_pct,
    t.forecast_cpu_pct,
    t.forecast_memory_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
WHERE t.cpu_utilization_pct >= 85
   OR t.memory_utilization_pct >= 85
   OR t.p95_latency_ms >= 500
   OR t.error_rate_pct >= 2
   OR t.forecast_cpu_pct >= 85
   OR t.forecast_memory_pct >= 85
ORDER BY
    t.sampled_at,
    s.service_name,
    t.host_id;
"""

run_sql(sql)


## 05.7 Question 5 — Daily service capacity trend

`DATE_TRUNC('day')` turns raw timestamps into daily buckets.
This is useful for trend reporting and capacity planning.


In [ ]:
sql = """
SELECT
    DATE_TRUNC('day', t.sampled_at) AS sample_day,
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(t.cpu_utilization_pct), 2) AS max_cpu_pct,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(MAX(t.memory_utilization_pct), 2) AS max_memory_pct,
    ROUND(AVG(t.error_rate_pct), 3) AS avg_error_rate_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY
    DATE_TRUNC('day', t.sampled_at),
    s.service_name
ORDER BY
    sample_day,
    s.service_name;
"""

run_sql(sql)


## 05.8 Question 6 — Hourly service capacity trend

Hourly buckets are useful when telemetry is collected every few minutes and we want operational trends.


In [ ]:
sql = """
SELECT
    DATE_TRUNC('hour', t.sampled_at) AS sample_hour,
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(t.cpu_utilization_pct), 2) AS max_cpu_pct,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(MAX(t.memory_utilization_pct), 2) AS max_memory_pct,
    ROUND(AVG(t.requests_per_min), 0) AS avg_requests_per_min,
    ROUND(AVG(t.error_rate_pct), 3) AS avg_error_rate_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY
    DATE_TRUNC('hour', t.sampled_at),
    s.service_name
ORDER BY
    sample_hour,
    s.service_name;
"""

run_sql(sql)


## 05.9 Question 7 — Which services look overallocated?

This compares allocated resources against actual usage.
It is useful for cost optimization and capacity rightsizing.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    s.service_name,
    t.host_id,
    t.allocated_cpu_cores,
    t.actual_cpu_cores,
    ROUND(t.allocated_cpu_cores - t.actual_cpu_cores, 2) AS unused_cpu_cores,
    t.allocated_memory_gb,
    t.actual_memory_gb,
    ROUND(t.allocated_memory_gb - t.actual_memory_gb, 2) AS unused_memory_gb,
    t.cloud_cost_usd
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
WHERE t.allocated_cpu_cores > t.actual_cpu_cores
   OR t.allocated_memory_gb > t.actual_memory_gb
ORDER BY
    t.cloud_cost_usd DESC,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 05.10 Question 8 — Which service costs the most?

If `cloud_cost_usd` is incremental per sample, SUM gives total cost over the period.
If it is a snapshot value, AVG or MAX may be better.
For this practice, treat it as incremental sample cost.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(SUM(t.cloud_cost_usd), 2) AS total_cloud_cost_usd,
    ROUND(AVG(t.cloud_cost_usd), 4) AS avg_sample_cloud_cost_usd,
    COUNT(*) AS sample_count
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY total_cloud_cost_usd DESC;
"""

run_sql(sql)


## 05.11 Question 9 — Risky hourly windows using a CTE

A CTE makes interview SQL easier to read.
First create the hourly rollup, then filter risky windows from the rollup.


In [ ]:
sql = """
WITH hourly_service_rollup AS (
    SELECT
        DATE_TRUNC('hour', t.sampled_at) AS sample_hour,
        s.service_name,
        ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
        ROUND(MAX(t.cpu_utilization_pct), 2) AS max_cpu_pct,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.cpu_utilization_pct)::NUMERIC,
            2
        ) AS p95_cpu_pct,
        ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct,
        ROUND(MAX(t.memory_utilization_pct), 2) AS max_memory_pct,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.memory_utilization_pct)::NUMERIC,
            2
        ) AS p95_memory_pct,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.p95_latency_ms)::NUMERIC,
            2
        ) AS p95_of_sampled_p95_latency_ms,
        ROUND(AVG(t.error_rate_pct), 3) AS avg_error_rate_pct
    FROM telemetry_samples t
    JOIN services s
        ON s.service_id = t.service_id
    GROUP BY
        DATE_TRUNC('hour', t.sampled_at),
        s.service_name
)
SELECT *
FROM hourly_service_rollup
WHERE p95_cpu_pct >= 85
   OR p95_memory_pct >= 85
   OR p95_of_sampled_p95_latency_ms >= 450
   OR avg_error_rate_pct >= 1.0
ORDER BY
    sample_hour,
    p95_cpu_pct DESC,
    p95_of_sampled_p95_latency_ms DESC;
"""

run_sql(sql)


## 05.12 Question 10 — Rank services by hourly CPU risk

This combines CTEs and window functions.
First aggregate service/hour metrics.
Then rank services inside each hour.


In [ ]:
sql = """
WITH hourly_service_rollup AS (
    SELECT
        DATE_TRUNC('hour', t.sampled_at) AS sample_hour,
        s.service_name,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.cpu_utilization_pct)::NUMERIC,
            2
        ) AS p95_cpu_pct
    FROM telemetry_samples t
    JOIN services s
        ON s.service_id = t.service_id
    GROUP BY
        DATE_TRUNC('hour', t.sampled_at),
        s.service_name
),
ranked AS (
    SELECT
        sample_hour,
        service_name,
        p95_cpu_pct,
        RANK() OVER (
            PARTITION BY sample_hour
            ORDER BY p95_cpu_pct DESC
        ) AS cpu_risk_rank
    FROM hourly_service_rollup
)
SELECT *
FROM ranked
WHERE cpu_risk_rank = 1
ORDER BY sample_hour;
"""

run_sql(sql)


## 05.13 Question 11 — Compare service CPU to previous hour

LAG compares the current row to a previous row.
Here we compare each service's current hourly P95 CPU to the previous hour.


In [ ]:
sql = """
WITH hourly_service_rollup AS (
    SELECT
        DATE_TRUNC('hour', t.sampled_at) AS sample_hour,
        s.service_name,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.cpu_utilization_pct)::NUMERIC,
            2
        ) AS p95_cpu_pct
    FROM telemetry_samples t
    JOIN services s
        ON s.service_id = t.service_id
    GROUP BY
        DATE_TRUNC('hour', t.sampled_at),
        s.service_name
),
with_previous AS (
    SELECT
        sample_hour,
        service_name,
        p95_cpu_pct,
        LAG(p95_cpu_pct) OVER (
            PARTITION BY service_name
            ORDER BY sample_hour
        ) AS previous_hour_p95_cpu_pct
    FROM hourly_service_rollup
)
SELECT
    sample_hour,
    service_name,
    p95_cpu_pct,
    previous_hour_p95_cpu_pct,
    ROUND(
        p95_cpu_pct - previous_hour_p95_cpu_pct,
        2
    ) AS p95_cpu_change
FROM with_previous
ORDER BY
    service_name,
    sample_hour;
"""

run_sql(sql)


## 05.14 Question 12 — Use JSONB tags to filter telemetry

The tags column stores flexible metadata.
Use `tags ->> 'region'` or `tags ->> 'service'` to extract JSON values as text.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    s.service_name,
    t.host_id,
    t.cpu_utilization_pct,
    t.memory_utilization_pct,
    t.tags,
    t.tags ->> 'region' AS tag_region,
    t.tags ->> 'service' AS tag_service
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
WHERE t.tags ? 'region'
ORDER BY
    t.sampled_at,
    s.service_name
LIMIT 50;
"""

run_sql(sql)


## 05.15 Optional incident/deployment questions

These are included because both `incidents` and `deployments` tables exist with usable columns in the current database.


### 05.15.a Incident window telemetry

This query joins incident timing to nearby telemetry windows so you can inspect metric behavior before/during incidents.


In [ ]:
sql = """
SELECT
    i.incident_id,
    s.service_name,
    i.started_at,
    i.ended_at,
    t.sampled_at,
    t.cpu_utilization_pct,
    t.memory_utilization_pct,
    t.p95_latency_ms,
    t.error_rate_pct
FROM incidents i
JOIN services s
    ON s.service_id = i.service_id
JOIN telemetry_samples t
    ON t.service_id = i.service_id
WHERE t.sampled_at BETWEEN i.started_at - INTERVAL '2 hour'
                      AND COALESCE(i.ended_at, i.started_at + INTERVAL '30 min') + INTERVAL '2 hour'
ORDER BY
    i.incident_id,
    t.sampled_at;
"""

run_sql(sql)


### 05.15.b Deployment before/after comparison

This query compares telemetry averages in windows before and after each service's latest deployment.


In [ ]:
sql = """
WITH latest_deploy AS (
    SELECT service_id, MAX(deployed_at) AS deployed_at
    FROM deployments
    GROUP BY service_id
)
SELECT
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct) FILTER (
        WHERE t.sampled_at >= d.deployed_at - INTERVAL '12 hour'
          AND t.sampled_at < d.deployed_at
    ), 2) AS cpu_before,
    ROUND(AVG(t.cpu_utilization_pct) FILTER (
        WHERE t.sampled_at >= d.deployed_at
          AND t.sampled_at < d.deployed_at + INTERVAL '12 hour'
    ), 2) AS cpu_after,
    ROUND(AVG(t.p95_latency_ms) FILTER (
        WHERE t.sampled_at >= d.deployed_at - INTERVAL '12 hour'
          AND t.sampled_at < d.deployed_at
    ), 2) AS latency_before,
    ROUND(AVG(t.p95_latency_ms) FILTER (
        WHERE t.sampled_at >= d.deployed_at
          AND t.sampled_at < d.deployed_at + INTERVAL '12 hour'
    ), 2) AS latency_after
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
JOIN latest_deploy d
    ON d.service_id = s.service_id
GROUP BY s.service_name
ORDER BY s.service_name;
"""

run_sql(sql)


## 05.16 Final interview explanation

For interview SQL, I start by clarifying the metric grain and the question. If I need service-level risk, I join telemetry samples to services and group by service. If I need time trends, I use DATE_TRUNC to create hourly or daily buckets. If I need readable multi-step logic, I use a CTE. If I need ranking or previous-period comparison, I use window functions like RANK and LAG. For capacity work, AVG shows normal usage, MAX shows spikes, and P95 shows sustained high pressure while reducing the effect of one-off outliers.
